In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

from load import load_data, tensorize_data, clean_data
from models import NaiveRNN, NaiveLSTM
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
def prepareDataLoader(dataset, labels, batchsize=32):
    train, test, train_labels, test_labels = train_test_split(dataset, labels, test_size=0.2, random_state=42)


    train_dataset = TensorDataset(train, train_labels)
    test_dataset = TensorDataset(test, test_labels)

    train_dataloader = DataLoader(train_dataset, batch_size=batchsize, shuffle=True)
    test_dataloader = DataLoader(test_dataset, batch_size=batchsize, shuffle=False)

    return (train_dataloader, test_dataloader)

In [ ]:
def training(epochs, model, dataloader, criterion, optimizer):
    for epoch in range(epochs):
        model.train()
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

def evaluate(model, dataloader):
    model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        print(f'Accuracy: {100 * correct / total}%')


In [4]:
dfs = load_data()

# The load_data function will return dfs in a different order. Parse through
for df in dfs:
    if df[1] == "PFC_con_4.csv":
        con4_df = df[0]
    if df[1] == "PFC_con_5.csv":
        con5_df = df[0]

In [ ]:
# Use DS+/DS- as labels (0)
tensors_4 = tensorize_data(con4_df, 0)
labels = tensors_4[1]
dataset = tensors_4[0]

# Unsqueeze my tensor bc it think I'm using a forward hook or something???
unsqueezed_tensor = dataset.unsqueeze(-1)

dataloaders = prepareDataLoader(unsqueezed_tensor, labels, 32)
train_dataloader = dataloaders[0]
test_dataloader = dataloaders[1]

In [6]:
rnn_model = NaiveRNN(1, 32, 2).to(device)
lstm_model = NaiveLSTM(1, 32, 2).to(device)

criterion = nn.CrossEntropyLoss()
rnn_optimizer = optim.Adam(rnn_model.parameters(), lr=0.001)
lstm_optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)


epochs = 50
training(epochs, rnn_model, train_dataloader, criterion, rnn_optimizer)
evaluate(rnn_model, test_dataloader)

Epoch [1/50], Loss: 0.6829
Epoch [2/50], Loss: 0.7001
Epoch [3/50], Loss: 0.7197
Epoch [4/50], Loss: 0.6993
Epoch [5/50], Loss: 0.6961
Epoch [6/50], Loss: 0.7058
Epoch [7/50], Loss: 0.7081
Epoch [8/50], Loss: 0.7045
Epoch [9/50], Loss: 0.6991
Epoch [10/50], Loss: 0.6873
Epoch [11/50], Loss: 0.6824
Epoch [12/50], Loss: 0.6853
Epoch [13/50], Loss: 0.7046
Epoch [14/50], Loss: 0.6960
Epoch [15/50], Loss: 0.6906
Epoch [16/50], Loss: 0.7019
Epoch [17/50], Loss: 0.6944
Epoch [18/50], Loss: 0.6883
Epoch [19/50], Loss: 0.6813
Epoch [20/50], Loss: 0.6766
Epoch [21/50], Loss: 0.6867
Epoch [22/50], Loss: 0.6863
Epoch [23/50], Loss: 0.6868
Epoch [24/50], Loss: 0.7056
Epoch [25/50], Loss: 0.7040
Epoch [26/50], Loss: 0.6644
Epoch [27/50], Loss: 0.6751
Epoch [28/50], Loss: 0.6877
Epoch [29/50], Loss: 0.7055
Epoch [30/50], Loss: 0.7190
Epoch [31/50], Loss: 0.6747
Epoch [32/50], Loss: 0.7271
Epoch [33/50], Loss: 0.6698
Epoch [34/50], Loss: 0.7054
Epoch [35/50], Loss: 0.6809
Epoch [36/50], Loss: 0.6800
E

In [7]:
con4_df_clean = clean_data(con4_df)
# Use DS+/DS- as labels (0)
dataset, labels = tensorize_data(con4_df_clean, 0)

# Unsqueeze my tensor bc it think I'm using a forward hook or something???
unsqueezed_tensor = dataset.unsqueeze(-1)

dataloaders = prepareDataLoader(unsqueezed_tensor, labels, 32)
train_dataloader = dataloaders[0]
test_dataloader = dataloaders[1]

epochs = 20
rnn_model = NaiveRNN(1, 32, 2).to(device)
training(epochs, rnn_model, train_dataloader, criterion, rnn_optimizer)
evaluate(rnn_model, test_dataloader)

Epoch [1/20], Loss: 0.7073
Epoch [2/20], Loss: 0.7258
Epoch [3/20], Loss: 0.6892
Epoch [4/20], Loss: 0.7339
Epoch [5/20], Loss: 0.6790
Epoch [6/20], Loss: 0.6724
Epoch [7/20], Loss: 0.7207
Epoch [8/20], Loss: 0.7187
Epoch [9/20], Loss: 0.7117
Epoch [10/20], Loss: 0.6958
Epoch [11/20], Loss: 0.7295
Epoch [12/20], Loss: 0.7280
Epoch [13/20], Loss: 0.7008
Epoch [14/20], Loss: 0.6884
Epoch [15/20], Loss: 0.6614
Epoch [16/20], Loss: 0.7148
Epoch [17/20], Loss: 0.6927
Epoch [18/20], Loss: 0.6873
Epoch [19/20], Loss: 0.6733
Epoch [20/20], Loss: 0.6893
Accuracy: 51.65562913907285%


In [ ]:
ds_minus = con4_df_clean[con4_df_clean.iloc[:, 3] == 0]
dataset, labels = tensorize_data(ds_minus, 0)

unsqueezed_tensor = dataset.unsqueeze(-1)

train, test = prepareDataLoader(unsqueezed_tensor, labels, 32)

criterion = nn.CrossEntropyLoss()
rnn_optimizer = optim.Adam(rnn_model.parameters(), lr=0.01)
epochs = 20
rnn_model = NaiveRNN(1, 32, 2).to(device)


training(epochs, rnn_model, train_dataloader, criterion, rnn_optimizer)
evaluate(rnn_model, test_dataloader)

Epoch [1/20], Loss: 0.6744
Epoch [2/20], Loss: 0.6959
Epoch [3/20], Loss: 0.6949
Epoch [4/20], Loss: 0.6718
Epoch [5/20], Loss: 0.6905
Epoch [6/20], Loss: 0.6800
Epoch [7/20], Loss: 0.6795
Epoch [8/20], Loss: 0.6923
Epoch [9/20], Loss: 0.7256
Epoch [10/20], Loss: 0.6995
Epoch [11/20], Loss: 0.6623
Epoch [12/20], Loss: 0.6947
Epoch [13/20], Loss: 0.6687
Epoch [14/20], Loss: 0.7481
Epoch [15/20], Loss: 0.6897
Epoch [16/20], Loss: 0.7371
Epoch [17/20], Loss: 0.7760
Epoch [18/20], Loss: 0.7257
Epoch [19/20], Loss: 0.7475
Epoch [20/20], Loss: 0.7341
Accuracy: 48.383326840670044%


In [ ]:
ds_minus = con4_df_clean[con4_df_clean.iloc[:, 3] == 0]
dataset, labels = tensorize_data(ds_minus, 0)


In [ ]:
# Investigate why this is only achieving ~50% accuracy but my test is achieving like 77%

In [ ]:
import os
con4_df = pd.read_csv(os.path.join("./Neuron Data", "PFC_con_4.csv"))
con4_df = con4_df.apply(pd.to_numeric, errors='coerce')

con4_df = clean_data(con4_df)

# Drop rat number, cell number, trial number
con4_df = con4_df.drop(columns=con4_df.columns[:3])
con4_df.columns = range(len(con4_df.columns))

# Split by trial type
con4_minus = con4_df[con4_df.iloc[:, 0] == 0]
con4_minus = con4_minus.drop(columns=con4_minus.columns[0])

# Extract labels
con4_minus_labels = con4_minus.iloc[:, 0].tolist()
con4_minus_labels = torch.tensor(con4_minus_labels, dtype=torch.long)

con4_minus = con4_minus.drop(columns=con4_minus.columns[0])

# Convert to tensors
con4_minus_tensor = torch.tensor(con4_minus.to_numpy(), dtype=torch.float32)


In [48]:
from sklearn.model_selection import train_test_split

# Trying to unsqueeze the tensor
con4_minus_tensor = torch.tensor(con4_minus.to_numpy(), dtype=torch.float32)
con4_minus_tensor = con4_minus_tensor.unsqueeze(-1)

train, test = prepareDataLoader(con4_minus_tensor, con4_minus_labels, 32)

criterion = nn.CrossEntropyLoss()
rnn_model = NaiveRNN(1, 32, 2).to(device)
rnn_optimizer = optim.Adam(rnn_model.parameters(), lr=0.001)
epochs = 20

training(epochs, rnn_model, train, criterion, rnn_optimizer)
evaluate(rnn_model, test)

Epoch [1/20], Loss: 0.8769
Epoch [2/20], Loss: 0.3735
Epoch [3/20], Loss: 0.2306
Epoch [4/20], Loss: 0.4021
Epoch [5/20], Loss: 0.3579
Epoch [6/20], Loss: 0.6626
Epoch [7/20], Loss: 0.5661
Epoch [8/20], Loss: 0.3474
Epoch [9/20], Loss: 0.3734
Epoch [10/20], Loss: 0.4971
Epoch [11/20], Loss: 0.5009
Epoch [12/20], Loss: 0.3852
Epoch [13/20], Loss: 0.6425
Epoch [14/20], Loss: 0.6439
Epoch [15/20], Loss: 0.5179
Epoch [16/20], Loss: 1.0226
Epoch [17/20], Loss: 0.4253
Epoch [18/20], Loss: 0.9224
Epoch [19/20], Loss: 0.2448
Epoch [20/20], Loss: 0.3776
Accuracy: 77.9423226812159%


In [ ]:
# Repeat just using base functionality
con4_df = pd.read_csv(os.path.join("./Neuron Data", "PFC_con_4.csv"))
con4_df = clean_data(con4_df)

con4_tensor, con4_labels = tensorize_data(con4_df, 1)

con4_tensor_unsqueeze = con4_tensor.unsqueeze(-1)

train, test = prepareDataLoader(con4_tensor_unsqueeze, con4_labels, 32)

criterion = nn.CrossEntropyLoss()
rnn_model = NaiveRNN(1, 32, 2).to(device)
rnn_optimizer = optim.Adam(rnn_model.parameters(), lr=0.001)
epochs = 20

training(epochs, rnn_model, train, criterion, rnn_optimizer)
evaluate(rnn_model, test)

Epoch [1/20], Loss: 0.7084
Epoch [2/20], Loss: 0.6409
Epoch [3/20], Loss: 0.6480
Epoch [4/20], Loss: 0.6293
Epoch [5/20], Loss: 0.6736
Epoch [6/20], Loss: 0.6427
Epoch [7/20], Loss: 0.6450
Epoch [8/20], Loss: 0.6870
Epoch [9/20], Loss: 0.6491
Epoch [10/20], Loss: 0.6860
Epoch [11/20], Loss: 0.6453
Epoch [12/20], Loss: 0.7302
Epoch [13/20], Loss: 0.6383
Epoch [14/20], Loss: 0.6698
Epoch [15/20], Loss: 0.7182
Epoch [16/20], Loss: 0.6867
Epoch [17/20], Loss: 0.6777
Epoch [18/20], Loss: 0.7082
Epoch [19/20], Loss: 0.6285
Epoch [20/20], Loss: 0.6565
Accuracy: 61.19984417608103%
